<a href="https://colab.research.google.com/github/MukiiriKoome/Apollo-II-Mission-RAG/blob/main/exercise/solution/Solutions_Gemini_Final_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt install tesseract-ocr libtesseract-dev
!pip install -q -U google-generativeai chromadb pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following additional packages will be installed:
  libarchive-dev libleptonica-dev
The following NEW packages will be installed:
  libarchive-dev libleptonica-dev libtesseract-dev
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,744 kB of archives.
After this operation, 16.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libarchive-dev amd64 3.6.0-1ubuntu1.7 [582 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libleptonica-dev amd64 1.82.0-3build1 [1,562 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libtesseract-dev amd64 4.1.1-2.1build1 [1,600 kB]
Fetched 3,744 kB in 1s (2,695 kB/s)
Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 118243 files and directories currently i

In [2]:
import time
from tqdm import tqdm
import pathlib
import google.generativeai as genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import pandas as pd
from PIL import Image
import pytesseract
from IPython.display import Markdown

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

# Gemini Final Exercise

You're an astronomy student who's very curious about the Apollo 11 missions,
and through your research, you've found a lot of different types of data (otherwise known as multimodal) from NASA's
public archive.

1. Text: You have the full final NASA report post-mission, spanning over 300
pages of incredibly informative content that details a summary of everything
that happened as well as conclusions that NASA researchers and engineers
came to. For the sake of this exercise, we've selected 3 particularly interesting pages, and converted them to images (you'll see soon why).

2. Video: You also have several clips of the famous Neil Armstrong and Buzz Aldrin footage as they
first stepped onto the moon, containing highlights of their moonwalks as well
as raising the American flag.

3. Audio: Finally, you have highlights from the audio recorded throughout the
mission, which provides insights into how communication between the astronauts
occurred as well as from the astronauts to mission control.

Now, you want to search through and summarize this information for your
upcoming research paper. Using your newfound skills from this course, you
can accomplish this using Gemini! In particular, we will build a Retrieval Augmented Generation (RAG) system that you can directly interact with.

## Data Preparation

Before we begin, ensure that you've uploaded the resources.zip folder and unzipped it using the following command:

In [4]:
!wget -O resources.zip "https://video.udacity-data.com/topher/2024/June/66744e79_resources/resources.zip"

--2026-07-17 03:50:03--  https://video.udacity-data.com/topher/2024/June/66744e79_resources/resources.zip
Resolving video.udacity-data.com (video.udacity-data.com)... 104.18.39.85, 172.64.148.171, 2a06:98c1:3100::6812:2755, ...
Connecting to video.udacity-data.com (video.udacity-data.com)|104.18.39.85|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 286142532 (273M) [application/zip]
Saving to: ‘resources.zip’

resources.zip       100%[===================>] 272.89M  53.4MB/s    in 5.5s    

2026-07-17 03:50:09 (49.3 MB/s) - ‘resources.zip’ saved [286142532/286142532]



In [5]:
!unzip resources.zip

Archive:  resources.zip
   creating: resources/
  inflating: __MACOSX/._resources    
   creating: resources/video/
  inflating: __MACOSX/resources/._video  
  inflating: resources/.DS_Store     
  inflating: __MACOSX/resources/._.DS_Store  
   creating: resources/audio/
  inflating: __MACOSX/resources/._audio  
   creating: resources/text/
  inflating: __MACOSX/resources/._text  
  inflating: resources/video/Apollo11PlaqueComparison.mov  
  inflating: __MACOSX/resources/video/._Apollo11PlaqueComparison.mov  
  inflating: resources/video/Apollo11Intro.mov  
  inflating: __MACOSX/resources/video/._Apollo11Intro.mov  
  inflating: resources/video/Apollo11MoonwalkMontage.mov  
  inflating: __MACOSX/resources/video/._Apollo11MoonwalkMontage.mov  
  inflating: resources/video/OneSmallStepCompilation.mov  
  inflating: __MACOSX/resources/video/._OneSmallStepCompilation.mov  
  inflating: resources/video/RaisingTheAmericanFlag.mov  
  inflating: __MACOSX/resources/video/._RaisingTheAmericanFl


As we saw throughout this course, when working with different types of data,
we first need to parse it in a way that Gemini can understand. We will prepare our data by extracting all file names from the `resources` directory.

In [6]:
data_dir = pathlib.Path("resources/")
all_file_names = [str(file) for file in data_dir.rglob("*") if file.is_file() and not file.name.startswith('.')]

In [7]:
for file_name in all_file_names:
    print(file_name)

print(len(all_file_names))

resources/audio/Apollo11OnboardAudioHighlightClip3.mp3
resources/audio/Apollo11OnboardAudioHighlightClip1.mp3
resources/audio/Apollo11OnboardAudioHighlightClip2.mp3
resources/audio/Apollo11OnboardAudioHighlightClip4.mp3
resources/audio/Apollo11OnboardAudioHighlightClip5.mp3
resources/video/Apollo11PlaqueComparison.mov
resources/video/BuzzDescendsCompilation.mov
resources/video/RaisingTheAmericanFlag.mov
resources/video/Apollo11Intro.mov
resources/video/OneSmallStepCompilation.mov
resources/video/Apollo11MoonwalkMontage.mov
resources/text/images-333.jpg
resources/text/images-020.jpg
resources/text/images-023.jpg
14


You should expect to see 14 files.

## Retrieval Augmented Generation (RAG)

To showcase how we build a RAG, we will first build one for the Text case, and generalize it further after. Here is the general idea:
1. **Data Preparation** (done above): We first collected various types of data from NASA's public archive related to the Apollo 11 mission, including text, video, and audio files.
2. **Data Extraction and Summarization**: Extract the multimodal data from images, e.g. extract text from images using Optical Character Recognition (OCR), and use Gemini to generate summaries using a specialized prompt.
3. **Embedding Generation**: Convert the generated summaries into vector embeddings using Gemini's Text Embedding Model. These embeddings represent the summaries in a numerical format suitable for efficient similarity searches.
4. **Creating a Vector Database**: A Vector database was created to store the embeddings. This database facilitates fast and efficient retrieval of relevant documents based on similarity searches. We chose to use Chroma DB.
5. **Querying the RAG System**: For a given query, the system retrieves the most relevant documents (based on their embeddings) and generates a response using the retrieved documents as context.

Something important to note is that RAGs are usually used only when there is a surplus of data. In other words, if the data can't fit into the model prompt. In this case, the data we provided likely can fit into Gemini's 1 million token window, but for the sake of simplicity and restrictions of Google Colab's runtime, we opted to use a smaller set of data.

### Text

We will use Tesseract OCR (Optical Character Recognition) to extract text from images of the NASA report.

In [8]:
pytesseract.pytesseract.tesseract_cmd = (r'/usr/bin/tesseract')

Let's create a function to take in our images of a PDF, transcribe them into text, and summarize each of them.

In [9]:
def create_text_summary():
  path = pathlib.Path("resources/text")

  text_summary_prompt = f"""You are an assistant tailored for summarizing text for retrieval.
  These summaries will be turned into vector embeddings and used to retrieve the raw text.
  Give a concise summary of the text that is well optimized for retrieval. Here is the text."""

  images = []
  text_summaries = []

  for f in path.glob("*"):
    if f.is_dir() or f.name.startswith('.'):
      continue

    image = Image.open(f)
    response = model.generate_content([text_summary_prompt, pytesseract.image_to_string(image)])

    images.append(image)
    text_summaries.append(response.text)

  return images, text_summaries

In [11]:
safety_settings = [
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE",
    },
]

model = genai.GenerativeModel('models/gemini-2.5-flash', safety_settings=safety_settings)

In [12]:
image_files, text_summaries = create_text_summary()

Now, we can check out the generated summaries of the three pages we have!

In [13]:
for text_summary in text_summaries:
  print(text_summary)

This document outlines the ground rules and assumptions for an EPS (Electrical Power System) analysis of a lunar mission. It details operational parameters and durations for key spacecraft systems, including descent and ascent stage batteries, S-band equipment, rendezvous radar, PGNCS (Primary Navigation and Guidance Subsystem), and forward window heaters, covering phases from lunar orbit checkout to liftoff.
This document is the Apollo 11 Flight Plan (AS-506/CSM-107/LM-5), prepared by the Flight Planning Branch. It schedules crew activities and operations for a G Type Mission Lunar Landing, based on July 16, 1969 launch trajectory parameters. The plan is under the configuration control of the Crew Procedures Control Board (CPCB), which governs specific categories of proposed changes, coordinated by T. A. Guillory. W. J. North handles distribution requests.
This document outlines the Apollo 11 mission's launch and translunar coast phases. It details the July 16, 1969 launch (9:32 EDT),

We create the Chroma database using the generated summaries. You might be wondering what Vector DB and Chroma DB are.

**Vector Database**: A specialized database designed to store and manage high-dimensional vectors, which are numerical representations of data points. It allows efficient similarity searches to find vectors (and their corresponding data) that are close to a given query vector.

**Chroma DB**: An implementation of a vector database used to store and retrieve vector embeddings. These embeddings are generated from our summaries and allow us to perform efficient similarity searches.


In [20]:
class GeminiEmbeddingFunction(EmbeddingFunction):
  def __call__(self, input: Documents) -> Embeddings:
    model = 'gemini-embedding-2'
    title = "Custom query"
    return genai.embed_content(model=model,
                                content=input,
                                task_type="retrieval_document",
                                title=title)["embedding"]

In [21]:
def create_chroma_db(documents, name):
  chroma_client = chromadb.Client()
  db = chroma_client.get_or_create_collection(name=name, embedding_function=GeminiEmbeddingFunction())

  for i, d in enumerate(documents):
    db.add(
      documents=d,
      ids=str(i)
    )
  return db

In [22]:
text_db = create_chroma_db(text_summaries, "text_nasa")

/tmp/ipykernel_349/1925432264.py:3: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  db = chroma_client.get_or_create_collection(name=name, embedding_function=GeminiEmbeddingFunction())


Let's also take a peak at the `text_db` and ensure that embeddings were generated:

In [25]:
data = {
    'embeddings': list(text_db.peek()['embeddings']),
    'documents': text_db.peek()['documents']
}

df = pd.DataFrame(data)
df

,embeddings,documents
0,"[-0.014528650790452957, 0.0036186245270073414,...",This document outlines the ground rules and as...
1,"[-0.03488601744174957, 0.012669136747717857, -...",This document is the Apollo 11 Flight Plan (AS...
2,"[-0.040911391377449036, 0.0011933016357943416,...",This document outlines the Apollo 11 mission's...


You should see a column called `embeddings` with what are seemingly random values, but these values are actually high-dimensional vectors that represent the semantic meaning of your summaries.

Now let's actually try querying our information. We'll test a simple example like getting some file that has to do with the Apollo 11 Flight Plan.

In [26]:
def get_relevant_files(query, db):
  results = db.query(query_texts=[query], n_results=3)
  return results["ids"][0]

In [27]:
files = get_relevant_files("Apollo 11 Flight Plan", text_db)
print(files)

['1', '2', '0']


You should expect to see something like `['1', '0', '2']`. This means that the first entry in the `text_db` is most similar. If you look above at our `pd.DataFrame` output, the document with id 1 is the document about the Apollo 11 Flight Plan, so this is working as we expected!

### Video and Audio

Congrats! You've successfully built a working RAG for text. Now, let's extend this concept to Video and Audio, and build out some more complex queries. We'll begin by generalizing the above summary creation function to all sorts of modalities.

In [28]:
def create_summary(modality):
  path = data_dir / modality

  summary_prompt = f"""You are an assistant tailored for summarizing {modality} for retrieval.
  These summaries will be turned into vector embeddings and used to retrieve the raw {modality}.
  Give a concise summary of the {modality} that is well optimized for retrieval. Here is the {modality}."""

  files = []
  summaries = []

  for f in path.glob("*"):
    if f.is_dir() or f.name.startswith('.'):
      continue
    print(f)

    if modality == "text":
      file = Image.open(f)
      response = model.generate_content([summary_prompt, pytesseract.image_to_string(file)])

    else:
      file = genai.upload_file(f)

      while file.state.name == "PROCESSING":
        print("Waiting for video file upload...\n", end='')
        time.sleep(5)
        file = genai.get_file(file.name)

      response = model.generate_content([summary_prompt, file])

    files.append(file)
    summaries.append(response.text)

  return files, summaries

Now, we will create a folder with all of our data of different modalities. In particular, the first 5 are audio files, next 3 are text files, and final 6 are video files.

In [29]:
all_files = []
all_summaries = []

for modality_type in ["audio", "text", "video"]:
  files, summaries = create_summary(modality_type)
  all_files.extend(files)
  all_summaries.extend(summaries)

resources/audio/Apollo11OnboardAudioHighlightClip3.mp3
resources/audio/Apollo11OnboardAudioHighlightClip1.mp3
resources/audio/Apollo11OnboardAudioHighlightClip2.mp3
resources/audio/Apollo11OnboardAudioHighlightClip4.mp3
resources/audio/Apollo11OnboardAudioHighlightClip5.mp3
resources/text/images-333.jpg
resources/text/images-020.jpg
resources/text/images-023.jpg
resources/video/Apollo11PlaqueComparison.mov
Waiting for video file upload...
resources/video/BuzzDescendsCompilation.mov
Waiting for video file upload...
resources/video/RaisingTheAmericanFlag.mov
Waiting for video file upload...
resources/video/Apollo11Intro.mov
Waiting for video file upload...
Waiting for video file upload...
resources/video/OneSmallStepCompilation.mov
Waiting for video file upload...
resources/video/Apollo11MoonwalkMontage.mov
Waiting for video file upload...


In [30]:
db = create_chroma_db(all_summaries, "nasa")

/tmp/ipykernel_349/1925432264.py:3: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  db = chroma_client.get_or_create_collection(name=name, embedding_function=GeminiEmbeddingFunction())


Again, ensure that the embeddings were generated. Notice that now, we have audio, video, and text data.

In [32]:
data = {
    'embeddings': list(db.peek()['embeddings']),
    'documents': db.peek()['documents']
}

df = pd.DataFrame(data)
df

,embeddings,documents
0,"[-0.04571662098169327, -0.008569099940359592, ...",Astronauts on a space mission (likely Apollo) ...
1,"[-0.01664717309176922, 0.01305939257144928, 0....","This audio captures a technical discussion, li..."
2,"[-0.024582229554653168, -0.00387150258757174, ...",This audio captures real-time operational comm...
3,"[-0.029860004782676697, 0.00514431856572628, -...",Audio contains mission control-like communicat...
4,"[-0.04145318642258644, -0.0013086828403174877,...",The audio features a rocket launch or aerospac...
5,"[-0.005023561883717775, 0.004094769712537527, ...",This document outlines the ground rules and as...
6,"[-0.032157693058252335, 0.011359044350683689, ...","This document is the Apollo 11 Flight Plan, pr..."
7,"[-0.05424237996339798, 0.010454029776155949, -...",This text details the initial phases of a spac...
8,"[-0.03941325843334198, -0.003050359198823571, ...",This video provides a side-by-side comparison ...
9,"[-0.03227260336279869, -0.01427802350372076, -...",A comparison video showing Buzz Aldrin descend...


In [33]:
files = get_relevant_files("communication with Mission Control", db)
print(files)

['3', '2', '4']


Can we do more than just return the most relevant file? Yes we can! We can ask Gemini to return a response to the query using the files it thinks are most relevant, provide an answer and tell us what files it used! This is really exciting, and has vast applications in many industries.

In [34]:
def query_rag(query, db):
    files = get_relevant_files(query, db)
    prompt = [all_files[int(f)] for f in files]
    prompt.append("Generate a response to the query using the provided files. Here is the query.")
    prompt.append(query)
    return model.generate_content(prompt).text, [all_file_names[int(f)] for f in files]

In [35]:
for response in query_rag("Explain what happened with the Apollo 11 Mission.", db):
    print(response)

The Apollo 11 mission, launched on **July 16, 1969**, was a historic spaceflight that successfully landed the first humans on the Moon. Its primary objective was a "G Type Mission Lunar Landing," as indicated in the flight plan.

Here's a breakdown of the key events of the mission:

*   **Launch:** The Saturn V rocket carrying the Apollo 11 crew (Neil Armstrong, Buzz Aldrin, and Michael Collins) successfully lifted off from Kennedy Space Center.
*   **Lunar Landing:** After orbiting the Moon, the Lunar Module (nicknamed "Eagle"), with Armstrong and Aldrin aboard, separated from the Command/Service Module (which remained in lunar orbit with Collins). The Eagle successfully landed on the Moon's surface on **July 20, 1969**. Armstrong famously announced, "Tranquility Base, here, the Eagle has landed."
*   **Moonwalk:** Shortly after landing, Neil Armstrong became the first human to step onto the lunar surface, delivering the iconic phrase: "That's one small step for man, one giant leap fo

In [36]:
for response in query_rag("What happens at the Translunar Coast in the Mission Description?", db):
    print(response)

According to the Mission Description, the following major events occur during the Translunar Coast, after TLI (Trans-Lunar Injection) and prior to LOI (Lunar Orbit Insertion):

*   **Transposition, docking, and LM (Lunar Module) ejection**, including SIVB (Saturn IVB) photography.
*   **Separation from SIVB** and a CSM (Command/Service Module) evasive maneuver.
*   **SIVB propulsive venting of propellants** (slingshot).
*   **Two series of P23 cislunar navigation sightings**, star/earth horizon, consisting of five sets at 06:00 GET and five sets at 24:30 GET.
*   **Four midcourse corrections** which take place at TLI + 9, TLI + 24, LOI - 22 and LOI - 5 hours with ΔV nominally zero (See Table 1-1).
['resources/video/RaisingTheAmericanFlag.mov', 'resources/audio/Apollo11OnboardAudioHighlightClip4.mp3', 'resources/audio/Apollo11OnboardAudioHighlightClip3.mp3']
